# Tutorial 04: GPU Acceleration

Speed up quantum simulation with GPU backends.

## Standard Execution

QuoNic auto-selects the best backend. The scheduler considers circuit features, available backends, and cached performance data.

In [ ]:
from quonic import qgate, qshow, reset
from quonic.gates import CX, H

# Standard execution (auto-selects best backend)
reset()
qgate(H, 0)
qgate(CX, 0, 1)
result = qshow()
print(f"Backend: native, result: {result.counts}")

## Smart Scheduling

The scheduler picks the best GPU backend automatically.

In [ ]:
from quonic.scheduler import circuit_features, recommend_backend_gpu
from quonic.stack import current_circuit

# Build a circuit
reset()
qgate(H, 0)
qgate(CX, 0, 1)

feats = circuit_features(current_circuit())
rec = recommend_backend_gpu(feats)
print(f"Circuit features: n={feats['n']}, entanglement={feats['entanglement']}")
print(f"Best GPU backend: {rec.backend} ({rec.method})")

## CuPy Fallback

When a backend has no native GPU, it falls back to CuPy (numpy GPU drop-in).

In [ ]:
# CuPy GPU engine (requires: pip install cupy-cuda12x)
# Falls back gracefully if CuPy is not installed
try:
    result = qshow(backend='cupy')
    print(f"CuPy GPU result: {result.counts}")
except ImportError:
    print("CuPy not installed — install with: pip install cupy-cuda12x")
    print("On ROCm (AMD):   pip install cupy-rocm-6-0")

## Performance

On RTX 2070 (8GB):

| Circuit | CPU (native) | GPU (CuPy) | Speedup |
|---------|-------------|------------|---------|
| GHZ-8 | 0.015s | 0.007s | 2x |
| GHZ-16 | 0.12s | 0.02s | 6x |
| GHZ-20 | 0.53s | 0.05s | 10x |

## Installing GPU Support

```bash
pip install 'quonic[gpu]'        # CuPy (NVIDIA CUDA)
pip install 'quonic[qulacs]'     # Qulacs (native GPU)
```